In [1]:
import warnings
warnings.simplefilter(action='ignore')

import argparse
import os,cv2
import random,numpy,pandas
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

In [2]:
import argparse
import os
import random,numpy
import torch
import torch.nn as nn
import torch.nn.parallel
import torch.optim as optim
import torch.nn.functional as F
import torch.utils.data
import torchvision
import torchvision.datasets as dset
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
import torchvision.utils as vutils
from torchvision.models.feature_extraction import create_feature_extractor
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

In [3]:
seed = 999
print("Random Seed: ", seed)
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

Random Seed:  999


In [4]:
ngpu=1
ngf,nc = 3,3
ndf = 64

fsc_test={}

transform = transforms.Compose([
    transforms.Resize((450,450)),
    transforms.RandomRotation(15),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

for i in os.listdir('/kaggle/input/detect-ai-vs-human-generated-images-model/test'):
    fsc_test[i]=transform(Image.open(f'/kaggle/input/detect-ai-vs-human-generated-images-model/test/{i}'))

In [5]:
device = torch.device("cuda:0" if (torch.cuda.is_available() and ngpu > 0) else "cpu")

In [6]:
class EffnetModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        effnet = torchvision.models.efficientnet_v2_m(weights=torchvision.models.efficientnet.EfficientNet_V2_M_Weights.DEFAULT)
        self.model = create_feature_extractor(effnet, ['flatten'])
        self.nn_fracture = torch.nn.Sequential(
            torch.nn.Linear(1280, 1)
        )
    def forward(self, x):
        x = self.model(x)['flatten']
        x = self.nn_fracture(x)
        return x

In [7]:
EFF_NET = EffnetModel().float()
EFF_NET= nn.DataParallel(EFF_NET).to(device)
#EFF_NET
#EFF_NET.load_state_dict(torch.load("/kaggle/input/d/mafiosoquasar/fake-scene-classification-model-2/0.00015554654237348586 87.0.pth",weights_only=False,map_location=torch.device('cpu')))

Downloading: "https://download.pytorch.org/models/efficientnet_v2_m-dc08266a.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_v2_m-dc08266a.pth
100%|██████████| 208M/208M [00:01<00:00, 198MB/s] 


In [8]:
fsc_submission=pandas.read_csv("/kaggle/input/ai-vs-human-generated-dataset/test.csv")['id']

In [2]:
sub = [0]*len(fsc_submission)
file_ = ['0.0007992219179868698 5.pth'] # os.listdir('/kaggle/input/d/mafiosoquasar/fake-scene-classification-model-2/')\n\n
for j in file_:  
    print(j)
    z_add,z_total=0,0    
    EFF_NET.load_state_dict(torch.load(f"/kaggle/input/detect-ai-vs-human-generated-images-mode-2/{j}",weights_only=False, map_location=torch.device('cpu')))    
    for i,j in zip(fsc_submission[10000:],range(len(fsc_submission[10000:]))):
        #print(j)
        img=fsc_test[i.split('/')[1]].reshape((1, 3, 450, 450)).float().to(device)    
        sub[j] += EFF_NET(img).sigmoid().cpu().detach().numpy()

NameError: name 'fsc_submission' is not defined

In [1]:
submission=pandas.DataFrame({'id' : fsc_submission, 
                             'label' : numpy.array(sub)/len(file_)})
pandas.DataFrame(submission).to_csv(f"submission.csv", index=False)
pandas.DataFrame(submission)

NameError: name 'pandas' is not defined